In [1]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")


🚀 Environment: Local
📁 Working directory set to: /home/aidmantas/repos/computer-data-analysis-report
✅ Environment ready!


# Data Extraction: Financials & Infrastructure

This notebook extracts raw data from yfinance, FRED, and gridstatus.

In [2]:
import yfinance as yf
import pandas as pd
import sys
import os
from src.data.dvc_storage import store_dataframes_in_dvc

COMPANY_TICKERS = {
    "DLR": "Digital Realty Trust", "EQIX": "Equinix", "AMT": "American Tower",
    "MSFT": "Microsoft", "GOOGL": "Alphabet (Google)", "AMZN": "Amazon",
    "META": "Meta Platforms", "NVDA": "NVIDIA"
}

company_dfs = {}
for ticker, name in COMPANY_TICKERS.items():
    try:
        t = yf.Ticker(ticker)
        # INCREASE PERIOD to 5y to cover all promises
        h = t.history(period="5y")
        if not h.empty:
            h = h.copy()
            h["ticker"] = ticker
            h["company_name"] = name
            # RESET INDEX to preserve Date as a column
            h = h.reset_index()
            company_dfs[ticker] = h
    except Exception as e: print(f"Error {ticker}: {e}")

if company_dfs:
    combined = pd.concat(company_dfs.values(), ignore_index=True)
    os.makedirs('data/raw', exist_ok=True)
    combined.to_csv('data/raw/company_financials_expanded.csv', index=False)
    print(f"Saved {len(combined)} financial rows with Date column.")

Saved 10048 financial rows with Date column.


## SEC EDGAR Filings via edgartools

Search 10-K filings for data center buildout mentions.
Source: SEC EDGAR via `edgar` library (no API key needed).

In [ ]:
from edgar import set_identity, Companyimport pandas as pdimport reimport osset_identity('DataAnalysisResearch research@datacenter-ai.com')EDGAR_TICKERS = [    'DLR', 'EQIX', 'AMT', 'CCI', 'SBAC',    'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'IRM']def extract_tenk_mentions(ticker):    results = {'ticker': ticker, 'mw_mentions': [], 'sections': {}}    try:        c = Company(ticker)        tenk = c.latest_tenk        results['filing_date'] = tenk.filing_date        for sec in ['business', 'risk_factors', 'management_discussion']:            txt = getattr(tenk, sec, '') or ''            results['sections'][sec] = txt            mw_pat = re.compile(r'\d[\d,]+\s*(?:MW|megawatt)', re.IGNORECASE)            results['mw_mentions'].extend(mw_pat.findall(txt))    except Exception as e:        results['error'] = str(e)    return resultsall_results = []for ticker in EDGAR_TICKERS:    r = extract_tenk_mentions(ticker)    all_results.append(r)    print(f"{ticker}: {len(r['mw_mentions'])} MW mentions")print(f'Extracted {len(all_results)} tickers')os.makedirs('data/raw', exist_ok=True)pd.DataFrame(all_results).to_csv('data/raw/sec_edgar_mentions.csv', index=False)print('Saved: data/raw/sec_edgar_mentions.csv')

## SEC Subsidiaries (EX-21)

Extract legal entity names and jurisdictions from SEC EX-21 subsidiary exhibits.

In [ ]:
from edgar import set_identity, Companyimport pandas as pdimport osset_identity('DataAnalysisResearch research@datacenter-ai.com')US_STATES = {    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',    'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho',    'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana',    'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',    'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',    'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina',    'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',    'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas',    'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia',    'Wisconsin', 'Wyoming', 'District of Columbia'}all_subs = []for ticker in EDGAR_TICKERS:    try:        tenk = Company(ticker).latest_tenk        subs = tenk.subsidiaries        for sub in subs:            all_subs.append({                'ticker': ticker,                'entity_name': sub.name,                'jurisdiction': sub.jurisdiction,                'is_us': sub.jurisdiction in US_STATES            })        print(f"{ticker}: {len(subs)} subsidiaries")    except Exception as e:        print(f"{ticker}: error - {e}")df_subs = pd.DataFrame(all_subs)os.makedirs('data/raw', exist_ok=True)df_subs.to_csv('data/raw/sec_subsidiaries.csv', index=False)print(f"Saved: {len(df_subs)} rows ({df_subs['is_us'].sum()} US)")

## SEC Filing Text Features

Extract structured features from 10-K filing text: MW mentions, data center counts, global presence.

In [ ]:
import pandas as pdimport reimport osdf_mw = pd.read_csv('data/raw/sec_edgar_mentions.csv')def count_mw_total(mentions_str):    if pd.isna(mentions_str) or mentions_str == '[]':        return 0    import ast    try:        items = ast.literal_eval(mentions_str)        return len(items)    except:        return 0def count_dc_text(business_txt):    if pd.isna(business_txt):        return 0    return business_txt.lower().count('data center')GLOBAL_KEYWORDS = ['europe', 'asia', 'latin america', 'africa', 'australia', 'canada', 'brazil', 'singapore', 'germany', 'ireland', 'netherlands', 'united kingdom', 'japan', 'india']def count_global(txt):    if pd.isna(txt):        return 0    txt_lower = txt.lower()    return sum(1 for kw in GLOBAL_KEYWORDS if kw in txt_lower)df_feats = pd.DataFrame({    'ticker': df_mw['ticker'],    'filing_date': df_mw['filing_date'],    'mw_mentions': df_mw['mw_mentions'].apply(count_mw_total),    'dc_count_10k': df_mw['sections'].apply(lambda x: count_dc_text(str(x)[:50000] if pd.notna(x) else '')),    'global_regions': df_mw['sections'].apply(lambda x: count_global(str(x)[:50000] if pd.notna(x) else '')),    'ai_mentions': df_mw['sections'].apply(lambda x: str(x).lower().count('artificial intelligence') if pd.notna(x) else 0),    'renewable_mentions': df_mw['sections'].apply(lambda x: str(x).lower().count('renewable') if pd.notna(x) else 0),})df_feats.to_csv('data/raw/sec_text_features.csv', index=False)print(f"Saved: {len(df_feats)} tickers")print(df_feats.to_string())